📅 **论文年份 (Year):2017 年**  
*Attention Is All You Need (Transformer) — Vaswani et al.*

# Paper 13: Attention Is All You Need(第13篇论文:Attention Is All You Need)
## Vaswani et al. (2017)(Vaswani 等人(2017))

### The Transformer: Pure Attention Architecture(Transformer:纯注意力架构)

Revolutionary architecture that replaced RNNs with self-attention, enabling modern LLMs.

一种革命性的架构,用自注意力(self-attention)取代了 RNN,为现代大语言模型(LLM)奠定了基础。

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 在这篇论文之前,让计算机理解和生成语言主要靠 RNN(循环神经网络)。RNN 就像一个人逐字读书:必须读完第一个词才能读第二个词,速度慢,而且读到后面容易忘记前面的内容。这篇论文想解决的正是这两个痛点——处理长句子时"记不住"和"无法并行加速"。

**💡 主要贡献:** 作者提出了一种全新的架构 Transformer,并大胆宣称"注意力就是你所需要的一切"——完全抛弃 RNN,只用"注意力机制"来处理语言。所谓注意力,好比你读一句话时,眼睛可以同时扫过全句,并自动聚焦在与当前词最相关的其他词上,而不必逐字排队等待。

**🔧 方法:** 核心是"自注意力":句子中的每个词都会和句中所有词"互相打分",分数越高代表关系越密切,再按分数加权融合信息。为了从多个角度理解句子(比如语法关系、指代关系),模型使用"多头注意力",相当于请多位专家同时从不同角度审阅同一句话。由于所有词是同时处理的,模型还额外加入"位置编码"来告诉它词的先后顺序。整体采用编码器-解码器结构,可以充分利用 GPU 并行计算,训练速度大幅提升。

**🌟 意义:** 这是深度学习历史上最重要的论文之一。GPT、ChatGPT、BERT 等几乎所有现代大语言模型,都建立在 Transformer 之上——GPT 的全称"生成式预训练 Transformer"里就带着它的名字。可以说,没有这篇论文,就没有今天的 ChatGPT。理解了 Transformer,你就拿到了理解整个大模型时代的钥匙,这也是本阅读清单中承前启后的关键一篇。

## 🎯 核心结论 (Key Takeaways)

- **论文的核心宣言:完全抛弃循环,只靠注意力。** RNN 必须逐字串行处理、长句子容易"忘记开头",而 Transformer 让句中每个词同时与所有词"互相打分",任意两个位置之间都是直接连接——串行 O(n) 的等待变成了可完全并行的计算,训练速度大幅提升。

- **它是整个大模型时代的基石。** 原始的编码器-解码器 Transformer 用于机器翻译,后来"仅编码器"变成 BERT(双向理解)、"仅解码器"变成 GPT(自回归生成),ChatGPT、ViT、CLIP 等几乎都建立在这套架构之上。

- **本 notebook 用纯 NumPy 从零搭出了全部零件并验证其正确性:** 缩放点积注意力(验证每行注意力权重之和恰好为 1)、多头注意力(d_model=64 拆成 8 个头、每头 d_k=8,输入输出形状保持 (10, 64) 不变)、正弦/余弦位置编码、"先扩大再压缩"的前馈网络(64→256→64)、以及 LayerNorm(把均值约 5、标准差约 3 的输入拉回均值约 0、标准差约 1)。

- **注意力可视化实验直观展示了"多头"的价值:** 把 4 个头的注意力矩阵画成热力图,可以看到每个头的图案各不相同——不同的头确实在关注序列中不同的关系,这正是多头设计"请多位专家从不同角度审阅同一句话"的效果。

- **因果掩码实验揭示了 GPT 的关键一步:** 用上三角掩码遮住"未来"后,注意力矩阵从满矩阵变成下三角——每个位置只能看到自己和之前的词。正是这一处小改动,让 Transformer 能够逐词生成文本,防止训练时"偷看答案"。

- **带走一句话:** 让每个词直接"看到"整个句子并按相关性加权融合信息,再叠上残差连接与 LayerNorm 组成的标准积木块,就足以取代循环网络——"注意力就是你所需要的一切",理解了这一点,你就拿到了理解 GPT/BERT 的钥匙。

## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:处理"序列"当然要按顺序读——语言是一个词接一个词说出来的,模型也理应像 RNN 那样逐字处理。** 但这篇论文发现:把循环彻底扔掉、让所有词同时"互相打分"反而效果更好——不仅翻译质量创下新高,训练还因为可以完全并行而快了一个数量级。本 notebook 的自注意力实验就展示了这一点:整个 (10, 64) 的序列一次矩阵乘法全部算完,任意两个词之间都是"一步直达",没有任何逐字等待。

- **常识认为:词的先后顺序这么重要的信息,得靠专门的网络结构去学。** 但论文发现:只要把一组固定的正弦/余弦波"叠加"到词向量上就够了——完全不用学习,纯数学公式生成的位置编码,效果和学出来的差不多。notebook 中画出的位置编码热力图显示,不同频率的波纹组合起来,就像给每个位置发了一个独一无二的"条形码"。

- **常识认为:模型算出的注意力分数越"原汁原味"越好,除个 √d_k 这种小系数无关紧要。** 但实验发现:不除以 √d_k,点积会随维度增大而变大,把 softmax 推到梯度接近 0 的"饱和区",训练直接学不动——一个看似可有可无的缩放因子,竟是整个架构能否训练成功的命门。

- **常识认为:"Attention Is All You Need"这种标题是学术论文里少见的狂言,注意力顶多是 RNN 的辅助配件。** 但历史证明这句狂言是真的:notebook 里用纯 NumPy 搭出的这几个零件(注意力 + 位置编码 + 前馈网络 + LayerNorm),配上因果掩码实验中那个小小的上三角遮罩,就是 GPT/ChatGPT 的全部骨架——遮住"未来"逐词生成,催生了整个大模型时代。

#### 💻 代码解读

**做什么:** 导入本笔记本要用到的基础工具库,并固定随机数种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算库,用来做矩阵运算)和 `matplotlib.pyplot`(画图库,用来可视化注意力图案)。
- 调用 `np.random.seed(42)` 固定随机种子——就像掷骰子前把骰子"调成固定套路",保证每次生成的随机矩阵都一样,方便复现实验结果。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Scaled Dot-Product Attention(缩放点积注意力)

The fundamental building block:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

最基本的构建模块:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

#### 💻 代码解读

**做什么:** 实现 Transformer 最核心的部件——缩放点积注意力(Scaled Dot-Product Attention),并用随机数据测试和可视化注意力权重矩阵。

**怎么做:**
- 先定义 `softmax` 函数:把一排分数变成加起来等于 1 的"概率",内部先减去最大值(`x_max`)防止数值溢出,是数值稳定的写法。
- 定义 `scaled_dot_product_attention(Q, K, V, mask)`:先算 `Q` 和 `K` 转置的点积得到相似度分数 `scores`,再除以 `sqrt(d_k)` 进行"缩放"(防止分数太大导致 softmax 饱和);如果传入 `mask`,给被遮挡的位置加上 `-1e9`(相当于负无穷,softmax 后权重变成 0)。
- 对分数做 softmax 得到注意力权重 `attention_weights`,再用它对 `V` 加权求和得到输出——好比开会时按"相关程度"给每个人的发言分配权重,最后汇总成结论。
- 用长度为 5、维度为 8 的随机 `Q/K/V` 做测试,打印输出形状并验证每行注意力权重之和为 1。
- 最后用 `plt.imshow` 把 5×5 的注意力权重矩阵画成热力图,颜色越亮表示该 Query 越"关注"那个 Key 位置。


In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax"""
    # 数值稳定技巧:先减去每行最大值再取exp,防止e^x溢出;softmax结果不变
    # keepdims=True保留被归约的维度(长度变1),这样才能与原数组广播相减
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Q: Queries (seq_len_q, d_k)
    K: Keys (seq_len_k, d_k)
    V: Values (seq_len_v, d_v)
    mask: Optional mask (seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Compute attention scores
    # Q·K^T得到每对(query,key)的相似度,形状:(seq_len_q, seq_len_k)
    # 除以√d_k:防止维度大时点积过大、softmax进入饱和区导致梯度消失(论文核心设计)
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # Apply mask if provided (for causality or padding)
    # 被mask的位置(mask=1)加上-1e9,softmax后权重趋近于0,相当于"看不见"
    if mask is not None:
        scores = scores + (mask * -1e9)
    
    # Softmax to get attention weights
    # 沿key维度做softmax:每个query对所有key的注意力权重之和为1
    attention_weights = softmax(scores, axis=-1)
    
    # Weighted sum of values
    # 用注意力权重对V加权求和,形状:(seq_len_q, seq_len_k)×(seq_len_k, d_v)->(seq_len_q, d_v)
    output = np.dot(attention_weights, V)
    
    return output, attention_weights

# Test scaled dot-product attention
# 自注意力场景:Q、K、V都来自同一序列(这里用随机向量演示)
seq_len = 5
d_model = 8

Q = np.random.randn(seq_len, d_model)
K = np.random.randn(seq_len, d_model)
V = np.random.randn(seq_len, d_model)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Attention output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention weights sum (should be 1): {attn_weights.sum(axis=1)}")

# Visualize attention pattern
plt.figure(figsize=(8, 6))
plt.imshow(attn_weights, cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights Matrix')
plt.show()

## Multi-Head Attention(多头注意力)

Multiple attention "heads" attend to different aspects of the input:
$$\text{MultiHead}(Q,K,V) = \text{Concat}(head_1, ..., head_h)W^O$$

多个注意力"头"(head)分别关注输入的不同方面:
$$\text{MultiHead}(Q,K,V) = \text{Concat}(head_1, ..., head_h)W^O$$

#### 💻 代码解读

**做什么:** 实现多头注意力(Multi-Head Attention)类,让模型能同时从多个不同"视角"去关注序列中的信息。

**怎么做:**
- `__init__` 中先检查 `d_model` 能被 `num_heads` 整除,然后创建四个随机初始化的权重矩阵:`W_q`、`W_k`、`W_v`(分别把输入投影成查询、键、值)和 `W_o`(最后的输出投影)。
- `split_heads` 把 `(seq_len, d_model)` 的大矩阵切成 `(num_heads, seq_len, d_k)`——就像把一份大报告拆给多个小组,每组只看自己负责的那部分维度。
- `forward` 的流程:先用三个权重矩阵做线性投影,再切分成多个头;用 for 循环对每个头分别调用前面写好的 `scaled_dot_product_attention`,并把各头的注意力权重存进 `self.attention_weights`(后面可视化会用到)。
- 各头的输出用 `np.stack` 堆叠后,由 `combine_heads` 拼回 `(seq_len, d_model)` 的形状,最后过一遍 `W_o` 输出——相当于各小组汇报后再做一次总汇总。
- 测试部分:用 `d_model=64`、`num_heads=8` 创建 `mha`,对随机输入 `X` 做自注意力(Q、K、V 都是 X),打印输入输出形状和每个头的维度(64/8=8)。


In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        # d_model必须能被头数整除,才能平均切分给每个头
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        # 每个头的维度:总维度平均分配,多头总计算量与单头相同
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V for all heads (parallelized)
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        
        # Output projection
        self.W_o = np.random.randn(d_model, d_model) * 0.1
    
    def split_heads(self, x):
        """Split into multiple heads: (seq_len, d_model) -> (num_heads, seq_len, d_k)"""
        seq_len = x.shape[0]
        # 先reshape把最后一维拆成(num_heads, d_k):(seq_len, d_model)->(seq_len, num_heads, d_k)
        x = x.reshape(seq_len, self.num_heads, self.d_k)
        # transpose交换前两轴,让每个头拥有独立的(seq_len, d_k)矩阵,便于逐头计算注意力
        return x.transpose(1, 0, 2)
    
    def combine_heads(self, x):
        """Combine heads: (num_heads, seq_len, d_k) -> (seq_len, d_model)"""
        seq_len = x.shape[1]
        # split_heads的逆操作:先换回(seq_len, num_heads, d_k),再拼接成d_model维
        x = x.transpose(1, 0, 2)
        return x.reshape(seq_len, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        """
        Multi-head attention forward pass
        
        Q, K, V: (seq_len, d_model)
        """
        # Linear projections
        # 三个可学习的线性变换,把输入分别投影到查询/键/值空间,形状保持(seq_len, d_model)
        Q = np.dot(Q, self.W_q.T)
        K = np.dot(K, self.W_k.T)
        V = np.dot(V, self.W_v.T)
        
        # Split into multiple heads
        Q = self.split_heads(Q)  # (num_heads, seq_len, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Apply attention to each head
        head_outputs = []
        self.attention_weights = []
        
        # 每个头在自己的d_k维子空间独立做注意力,可以关注不同类型的关系
        for i in range(self.num_heads):
            head_out, head_attn = scaled_dot_product_attention(
                Q[i], K[i], V[i], mask
            )
            head_outputs.append(head_out)
            self.attention_weights.append(head_attn)
        
        # Stack heads
        heads = np.stack(head_outputs, axis=0)  # (num_heads, seq_len, d_k)
        
        # Combine heads
        combined = self.combine_heads(heads)  # (seq_len, d_model)
        
        # Final linear projection
        # W_o把拼接后的多头结果融合成最终输出,对应论文中的Concat(...)W^O
        output = np.dot(combined, self.W_o.T)
        
        return output

# Test multi-head attention
d_model = 64
num_heads = 8
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

X = np.random.randn(seq_len, d_model)
# Q=K=V=X即自注意力:序列内部每个位置互相关注
output = mha.forward(X, X, X)  # Self-attention

print(f"\nMulti-Head Attention:")
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Number of heads: {num_heads}")
print(f"Dimension per head: {mha.d_k}")

## Positional Encoding(位置编码)

Since Transformers have no recurrence, we add position information:
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

由于 Transformer 没有循环结构(recurrence),我们需要额外注入位置信息:
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

#### 💻 代码解读

**做什么:** 实现正弦/余弦位置编码(Positional Encoding),给"天生不知道顺序"的 Transformer 注入每个词的位置信息,并画图展示编码的样子。

**怎么做:**
- 定义 `positional_encoding(seq_len, d_model)`:先创建全零矩阵 `pe`,用 `position` 表示每个位置的序号,用 `div_term` 生成一组按指数递减的频率。
- 偶数维度(`pe[:, 0::2]`)填入 `sin` 值,奇数维度(`pe[:, 1::2]`)填入 `cos` 值——不同维度用不同频率的波,就像钟表的秒针、分针、时针:快慢不同的"指针"组合起来就能唯一确定一个时刻(位置)。
- 生成长度 50、维度 64 的位置编码,画两张图:上图用热力图展示所有维度的编码全貌(红蓝条纹),下图挑出第 0、1、2、3、10、20 这几个维度画曲线,可以直观看到前面的维度波动快、后面的维度波动慢。
- 最后打印编码矩阵形状,说明不同频率在不同尺度上编码位置。


In [ ]:
def positional_encoding(seq_len, d_model):
    """
    Create sinusoidal positional encoding
    """
    pe = np.zeros((seq_len, d_model))
    
    # [:, np.newaxis]把位置向量变成列向量(seq_len, 1),便于和div_term广播成矩阵
    position = np.arange(0, seq_len)[:, np.newaxis]
    # 等价于1/10000^(2i/d_model):用exp+log计算更稳定;不同维度对应不同频率的正弦波
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    # Apply sin to even indices
    # 切片0::2取偶数维;position*div_term广播:(seq_len,1)*(d_model/2,)->(seq_len, d_model/2)
    pe[:, 0::2] = np.sin(position * div_term)
    
    # Apply cos to odd indices
    # 奇数维用cos:sin/cos成对出现,使任意位置偏移都能表示为线性变换,便于模型学习相对位置
    pe[:, 1::2] = np.cos(position * div_term)
    
    return pe

# Generate positional encodings
seq_len = 50
d_model = 64
pe = positional_encoding(seq_len, d_model)

# Visualize positional encodings
plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.imshow(pe.T, cmap='RdBu', aspect='auto')
plt.colorbar(label='Encoding Value')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding (All Dimensions)')

plt.subplot(2, 1, 2)
# Plot first few dimensions
for i in [0, 1, 2, 3, 10, 20]:
    plt.plot(pe[:, i], label=f'Dim {i}')
plt.xlabel('Position')
plt.ylabel('Encoding Value')
plt.title('Positional Encoding (Selected Dimensions)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {pe.shape}")
print(f"Different frequencies encode position at different scales")

## Feed-Forward Network(前馈网络)

Applied to each position independently:
$$FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

对每个位置独立应用:
$$FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

#### 💻 代码解读

**做什么:** 实现 Transformer 中的前馈网络(FeedForward),它对序列中每个位置独立地做一次"先扩大再压缩"的非线性变换。

**怎么做:**
- `__init__` 创建两层的权重和偏置:`W1/b1` 把维度从 `d_model` 升到 `d_ff`,`W2/b2` 再降回 `d_model`。
- `forward` 中第一层算完后用 `np.maximum(0, ...)` 做 ReLU 激活(负数归零,只留正数),第二层做普通线性变换输出——相当于先把信息"摊开"到更大的空间加工,再压缩回原来的尺寸。
- 测试部分:`d_model=64`、`d_ff=256`(中间层通常是 4 倍宽),输入 10×64 的随机矩阵,打印输入、隐藏层、输出的形状,验证输出形状和输入一致。


In [ ]:
class FeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        # First layer with ReLU
        # np.maximum(0,·)就是ReLU;先升维:(seq_len, d_model)->(seq_len, d_ff)
        # FFN对每个位置独立施加同一变换,给模型提供非线性表达能力
        hidden = np.maximum(0, np.dot(x, self.W1) + self.b1)
        
        # Second layer
        # 再降回原维度:(seq_len, d_ff)->(seq_len, d_model),保证残差连接可以相加
        output = np.dot(hidden, self.W2) + self.b2
        
        return output

# Test feed-forward
d_model = 64
d_ff = 256  # Usually 4x larger

ff = FeedForward(d_model, d_ff)
x = np.random.randn(10, d_model)
output = ff.forward(x)

print(f"\nFeed-Forward Network:")
print(f"Input: {x.shape}")
print(f"Hidden: ({x.shape[0]}, {d_ff})")
print(f"Output: {output.shape}")

## Layer Normalization(层归一化)

Normalize across features (not batch like BatchNorm)

在特征维度上进行归一化(不同于 BatchNorm 在批次维度上归一化)

#### 💻 代码解读

**做什么:** 实现层归一化(LayerNorm),把每个位置的特征"拉平"到均值约 0、标准差约 1,让训练更稳定。

**怎么做:**
- `__init__` 创建两个可学习参数:缩放系数 `gamma`(初始为全 1)和平移系数 `beta`(初始为全 0),以及防止除零的小常数 `eps`。
- `forward` 沿最后一个维度(特征维)计算每个位置自己的均值 `mean` 和标准差 `std`,做标准化 `(x - mean) / (std + eps)`,再乘 `gamma` 加 `beta`——就像把每个学生的分数按自己班级的平均分和波动幅度换算成标准分。
- 注意它是对"每个样本自己的特征"做归一化,而不是像 BatchNorm 那样跨样本归一化。
- 测试部分:故意构造均值约 5、标准差约 3 的输入(`* 3 + 5`),经过 `ln.forward` 后打印对比,可以看到输出的均值接近 0、标准差接近 1。


In [ ]:
class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        # gamma/beta是可学习的缩放和平移参数,让网络能恢复任意需要的分布
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
    
    def forward(self, x):
        # LayerNorm沿特征维(最后一维)归一化,每个位置独立,与batch大小无关(区别于BatchNorm)
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        
        # 加eps防止std为0时除零
        normalized = (x - mean) / (std + self.eps)
        output = self.gamma * normalized + self.beta
        
        return output

ln = LayerNorm(d_model)
x = np.random.randn(10, d_model) * 3 + 5  # Unnormalized
normalized = ln.forward(x)

print(f"\nLayer Normalization:")
print(f"Input mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"Output mean: {normalized.mean():.4f}, std: {normalized.std():.4f}")

## Complete Transformer Block(完整的 Transformer 块)

#### 💻 代码解读

**做什么:** 把前面实现的所有零件组装成一个完整的 Transformer 块(TransformerBlock)——这就是 Transformer 网络反复堆叠的基本单元。

**怎么做:**
- `__init__` 组装四个组件:`MultiHeadAttention`(多头注意力)、两个 `LayerNorm`(`norm1`、`norm2`)和 `FeedForward`(前馈网络)。
- `forward` 分两步,每步都带"残差连接":第一步做自注意力,把注意力输出 `attn_output` 与原输入 `x` 相加后再过 `norm1`;第二步过前馈网络,把 `ff_output` 与输入相加后再过 `norm2`。
- 残差连接(`x + 输出`)就像"抄近道":即使中间层学得不好,原始信息也能直接传下去,让深层网络更容易训练。
- 测试部分:创建 `d_model=64`、8 个头、`d_ff=256` 的块,输入 10×64 的随机矩阵,验证输出形状不变,并打印这个块包含的四大部件清单。


In [ ]:
class TransformerBlock:
    def __init__(self, d_model, num_heads, d_ff):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Multi-head attention with residual connection
        attn_output = self.attention.forward(x, x, x, mask)
        # x+attn_output是残差连接:保留原始信息、缓解梯度消失;之后做LayerNorm(Post-LN结构)
        x = self.norm1.forward(x + attn_output)
        
        # Feed-forward with residual connection
        ff_output = self.ff.forward(x)
        # 第二个子层同样是"残差+归一化";堆叠N个这样的block就构成Transformer编码器
        x = self.norm2.forward(x + ff_output)
        
        return x

# Test transformer block
block = TransformerBlock(d_model=64, num_heads=8, d_ff=256)
x = np.random.randn(10, 64)
output = block.forward(x)

print(f"\nTransformer Block:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nBlock contains:")
print(f"  1. Multi-Head Self-Attention")
print(f"  2. Layer Normalization")
print(f"  3. Feed-Forward Network")
print(f"  4. Residual Connections")

## Visualize Multi-Head Attention Patterns(可视化多头注意力模式)

#### 💻 代码解读

**做什么:** 可视化多头注意力中每个头的注意力图案,直观展示"不同的头关注不同的模式"。

**怎么做:**
- 创建一个 `d_model=64`、`num_heads=4` 的 `MultiHeadAttention`,对长度为 8 的随机输入 `X` 做一次自注意力前向计算(`mha.forward(X, X, X)`)。
- 前向计算时每个头的注意力权重已经存在 `mha.attention_weights` 里,这里用 `plt.subplots` 创建 1 行 4 列的子图,循环把每个头的 8×8 注意力矩阵用 `imshow` 画成热力图(统一色标范围 0 到 1)。
- 每张子图横轴是 Key 位置、纵轴是 Query 位置,并加上共用的颜色条——对比四张图可以看到每个头的"关注方式"各不相同,就像四位审稿人各自盯着文章的不同方面。


In [ ]:
# Create attention with interpretable input
seq_len = 8
d_model = 64
num_heads = 4

mha = MultiHeadAttention(d_model, num_heads)
X = np.random.randn(seq_len, d_model)
output = mha.forward(X, X, X)

# Plot attention patterns for each head
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

# enumerate同时取头的编号i和对应子图ax;forward时保存的attention_weights按头索引
for i, ax in enumerate(axes):
    attn = mha.attention_weights[i]
    im = ax.imshow(attn, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')
    
plt.colorbar(im, ax=axes, label='Attention Weight', fraction=0.046, pad=0.04)
plt.suptitle('Multi-Head Attention Patterns', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("\nEach head learns to attend to different patterns!")
print("Different heads capture different relationships in the data.")

## Causal (Masked) Self-Attention for Autoregressive Models(用于自回归模型的因果(掩码)自注意力)

#### 💻 代码解读

**做什么:** 实现因果掩码(causal mask),对比"能看到未来"的双向注意力和"只能看过去"的因果注意力,这是 GPT 这类自回归语言模型的关键机制。

**怎么做:**
- 定义 `create_causal_mask(seq_len)`:用 `np.triu(..., k=1)` 生成一个上三角矩阵(对角线以上全是 1),1 代表"被遮挡、不许看"——即每个位置不能偷看它后面的词,就像做完形填空时把答案后面的内容盖住。
- 用同一组随机 `Q/K/V` 分别调用 `scaled_dot_product_attention` 两次:一次不带掩码(得到双向注意力 `attn_bi`),一次传入 `causal_mask`(得到因果注意力 `attn_causal`)。
- 画三张并排的图:左边是因果掩码本身(红色区域为禁区),中间是双向注意力(整个矩阵都有权重),右边是因果注意力(只有下三角有权重,上三角全为 0)。
- 最后打印说明:因果掩码用于自回归生成(如 GPT),防止未来信息泄露,让每个位置只能关注自己和之前的位置。


In [ ]:
def create_causal_mask(seq_len):
    """Create mask to prevent attending to future positions"""
    # np.triu取上三角矩阵,k=1表示从主对角线上方一格开始:未来位置标1(要屏蔽),自身及过去为0
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    return mask

# Test causal attention
seq_len = 8
causal_mask = create_causal_mask(seq_len)

Q = np.random.randn(seq_len, d_model)
K = np.random.randn(seq_len, d_model)
V = np.random.randn(seq_len, d_model)

# Without mask (bidirectional)
output_bi, attn_bi = scaled_dot_product_attention(Q, K, V)

# With causal mask (unidirectional)
# 加上因果mask后,注意力矩阵变成下三角:每个位置只能看到自己和之前的token(GPT式解码)
output_causal, attn_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# Visualize difference
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Causal mask
ax1.imshow(causal_mask, cmap='Reds', aspect='auto')
ax1.set_title('Causal Mask\n(1 = masked/not allowed)')
ax1.set_xlabel('Key Position')
ax1.set_ylabel('Query Position')

# Bidirectional attention
im2 = ax2.imshow(attn_bi, cmap='viridis', aspect='auto', vmin=0, vmax=1)
ax2.set_title('Bidirectional Attention\n(can see future)')
ax2.set_xlabel('Key Position')
ax2.set_ylabel('Query Position')

# Causal attention
im3 = ax3.imshow(attn_causal, cmap='viridis', aspect='auto', vmin=0, vmax=1)
ax3.set_title('Causal Attention\n(cannot see future)')
ax3.set_xlabel('Key Position')
ax3.set_ylabel('Query Position')

plt.colorbar(im3, ax=[ax2, ax3], label='Attention Weight')
plt.tight_layout()
plt.show()

print("\nCausal masking is crucial for:")
print("  - Autoregressive generation (GPT, language models)")
print("  - Prevents information leakage from future tokens")
print("  - Each position can only attend to itself and previous positions")

## Key Takeaways(核心要点)

### Why "Attention Is All You Need"?(为什么说"注意力就是你所需要的一切"?)
- **No recurrence**: Processes entire sequence in parallel
- **No convolution**: Pure attention mechanism
- **Scales better**: O(n²d) vs O(n) sequential operations in RNNs
- **Long-range dependencies**: Direct connections between any positions

- **无循环结构**:并行处理整个序列
- **无卷积**:纯粹的注意力(attention)机制
- **扩展性更好**:O(n²d) 的并行计算,而 RNN 需要 O(n) 的串行操作
- **长距离依赖**:任意位置之间都有直接连接

### Core Components:(核心组件:)
1. **Scaled Dot-Product Attention**: Efficient attention computation
2. **Multi-Head Attention**: Multiple representation subspaces
3. **Positional Encoding**: Inject position information
4. **Feed-Forward Networks**: Position-wise transformations
5. **Layer Normalization**: Stabilize training
6. **Residual Connections**: Enable deep networks

1. **缩放点积注意力(Scaled Dot-Product Attention)**:高效的注意力计算
2. **多头注意力(Multi-Head Attention)**:多个表示子空间
3. **位置编码(Positional Encoding)**:注入位置信息
4. **前馈网络(Feed-Forward Networks)**:逐位置的变换
5. **层归一化(Layer Normalization)**:稳定训练过程
6. **残差连接(Residual Connections)**:使深层网络得以训练

### Architecture Variants:(架构变体:)
- **Encoder-Decoder**: Original Transformer (translation)
- **Encoder-only**: BERT (bidirectional understanding)
- **Decoder-only**: GPT (autoregressive generation)

- **编码器-解码器(Encoder-Decoder)**:原始 Transformer(机器翻译)
- **仅编码器(Encoder-only)**:BERT(双向理解)
- **仅解码器(Decoder-only)**:GPT(自回归生成)

### Advantages:(优势:)
- Parallelizable training (unlike RNNs)
- Better long-range dependencies
- Interpretable attention patterns
- State-of-the-art on many tasks

- 训练可并行化(不同于 RNN)
- 更好地捕捉长距离依赖
- 注意力模式可解释
- 在众多任务上达到最先进(state-of-the-art)水平

### Impact:(影响:)
- Foundation of modern NLP: GPT, BERT, T5, etc.
- Extended to vision: Vision Transformer (ViT)
- Multi-modal models: CLIP, Flamingo
- Enabled LLMs with billions of parameters

- 现代 NLP 的基石:GPT、BERT、T5 等
- 扩展到视觉领域:视觉 Transformer(ViT)
- 多模态模型:CLIP、Flamingo
- 使数十亿参数规模的大语言模型(LLM)成为可能

## Companion Resource(配套资源)

**The Annotated Transformer** — Rush (2018)
https://nlp.seas.harvard.edu/annotated-transformer/

A line-by-line PyTorch annotation of this paper. Read alongside this notebook to see how the same architecture maps to a production framework. Where this notebook implements attention in pure NumPy to expose the mechanics, Rush's annotation shows how the same operations are expressed in a real training codebase.

**The Annotated Transformer(带注释的 Transformer)** — Rush(2018)

这是对本论文逐行进行 PyTorch 注释的实现。可与本笔记本对照阅读,了解同一架构如何映射到生产级框架。本笔记本用纯 NumPy 实现注意力(attention)以揭示其内部机制,而 Rush 的注释版则展示了同样的运算在真实训练代码库中的表达方式。